In [3]:
import pandas as pd
import json
import os

# Using ../ because the notebook is inside the 'notebooks' folder
df_clean = pd.read_csv('../data/processed/cleaned_transactions.csv')
# Ensure InvoiceDate is datetime for the report
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"Cleaned Dataset Loaded: {df_clean.shape[0]} rows")

Cleaned Dataset Loaded: 342273 rows


In [4]:
# CHECK 1: No missing values
assert df_clean.isnull().sum().sum() == 0, "❌ Missing values found!"

# CHECK 2: All quantities positive
assert (df_clean['Quantity'] > 0).all(), "❌ Negative quantities found!"

# CHECK 3: All prices positive
assert (df_clean['UnitPrice'] > 0).all(), "❌ Invalid prices found!"

# CHECK 4: CustomerID all integers
# Note: Since CSVs load as float if there were NaNs, we ensure it's int here
df_clean['CustomerID'] = df_clean['CustomerID'].astype('int64')
assert df_clean['CustomerID'].dtype == 'int64', "❌ CustomerID not integer!"

# CHECK 5: Date range valid
assert df_clean['InvoiceDate'].min() < df_clean['InvoiceDate'].max(), "❌ Invalid date range!"

print("✅ All mandatory validation checks passed!")

✅ All mandatory validation checks passed!


In [5]:
# Calculate metrics for the JSON report
validation_report = {
    'total_rows': len(df_clean),
    'total_columns': len(df_clean.columns),
    'date_range': f"{df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}",
    'unique_customers': int(df_clean['CustomerID'].nunique()),
    'unique_products': int(df_clean['StockCode'].nunique()),
    'unique_countries': int(df_clean['Country'].nunique()),
    'total_revenue': float(df_clean['TotalPrice'].sum()),
    'average_order_value': float(df_clean.groupby('InvoiceNo')['TotalPrice'].sum().mean()),
    'validation_passed': True
}

# Save to processed folder
output_path = '../data/processed/validation_report.json'
with open(output_path, 'w') as f:
    json.dump(validation_report, f, indent=4, default=str)

print(f"✅ Validation report saved to {output_path}")
# Display the report for confirmation
print(json.dumps(validation_report, indent=4))

✅ Validation report saved to ../data/processed/validation_report.json
{
    "total_rows": 342273,
    "total_columns": 13,
    "date_range": "2009-12-01 07:45:00 to 2010-12-09 20:01:00",
    "unique_customers": 4140,
    "unique_products": 3644,
    "unique_countries": 37,
    "total_revenue": 4479987.023999998,
    "average_order_value": 260.22229460966537,
    "validation_passed": true
}
